In [1]:
import pandas as pd
import json
import os
import deepsig
from IPython.display import display

In [2]:
cols = ['dataset', 'method', 'fitness_rule', 'fitness', 'ACC', 'MCC', 'f1_score', 'avg_odds_diff', 'stat_par_diff', 'eq_opp_diff']

In [3]:
def read_csv_files_from_folder(folder_path):
    dfs = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(folder_path, file_name)
            dfs.append(pd.read_csv(file_path))
    return pd.concat(dfs, ignore_index=True)

mlp_baseline_results = pd.read_csv('simple_mlp_results.csv')
mlp_baseline_results.replace({'simple_mlp_initializer': 'MLP'}, inplace=True)

mlp_xi_reg_results = pd.read_csv('mlp_xi_reg_results.csv')
mlp_xi_reg_results.replace({'mlp_xi_reg_initializer': r'MLP+SDR$_{\xi}$'}, inplace=True)

mlp_results = pd.concat([mlp_baseline_results, mlp_xi_reg_results])

ftl_baseline_results = pd.read_csv('ftl_mlp_results.csv')
ftl_baseline_results.replace({'ftl_mlp_initializer': 'FTL'}, inplace=True)

ftl_xi_reg_results = pd.read_csv('ftl_mlp_xi_reg_results.csv')
ftl_xi_reg_results.replace({'ftl_mlp_xi_reg_initializer': r'FTL+SDR$_{\xi}$'}, inplace=True)

ftl_results = pd.concat([ftl_baseline_results, ftl_xi_reg_results])

# HIFI results are stored as raw per-run CSVs produced by ablation.py's hifi_initializer
hifi_result = pd.read_csv('hifi_results.csv')
hifi_result.replace({'hifi_initializer': 'HIFI'}, inplace=True)

hifi_results = pd.concat([hifi_result, ftl_xi_reg_results])

full_results = pd.concat([mlp_results, ftl_results, hifi_results])

In [4]:
for results in [mlp_results,ftl_results,hifi_results, full_results]:
    results.replace({'adult_dataset_reader': 'Adult Income', 'compas_dataset_reader': 'Compas Recidivism', 'german_dataset_reader': 'German Credit', 'bank_dataset_reader': 'Bank Marketing'}, inplace=True)
    results.rename(columns={'avg_odds_diff': 'Equalized Odds', 'stat_par_diff': 'Statistical Parity', 'eq_opp_diff': 'Equal Opportunity', 'MCC': 'Mathew Correlation', 'ACC': 'Accuracy'}, inplace=True)

In [5]:
fitness_rules_target_metrics = {
    'mcc_parity': {'performance': 'Mathew Correlation', 'fairness': 'Statistical Parity'},
    'mcc_opportunity': {'performance': 'Mathew Correlation', 'fairness': 'Equal Opportunity'},
    'mcc_odds': {'performance': 'Mathew Correlation', 'fairness': 'Equalized Odds'},
    'acc_parity': {'performance': 'Accuracy', 'fairness': 'Statistical Parity'},
    'acc_opportunity': {'performance': 'Accuracy', 'fairness': 'Equal Opportunity'},
    'acc_odds': {'performance': 'Accuracy', 'fairness': 'Equalized Odds'}
}

fitness_rules_target_metrics = {
    'mcc_parity': ('Mathew Correlation', 'Statistical Parity'),
    'mcc_opportunity': ('Mathew Correlation', 'Equal Opportunity'),
    'mcc_odds': ('Mathew Correlation', 'Equalized Odds'),
    'acc_parity': ('Accuracy', 'Statistical Parity'),
    'acc_opportunity': ('Accuracy', 'Equal Opportunity'),
    'acc_odds': ('Accuracy', 'Equalized Odds')
}
fitness_rules_abvr = {
    'mcc_parity': 'Max(MCC - Stat. Parity)',
    'mcc_opportunity': 'Max(MCC - Eq. Odds)',
    'mcc_odds': 'Max(MCC - Eq. Opp.)',
    'acc_parity': 'Max(Acc - Stat. Parity)',
    'acc_opportunity': 'Max(Acc - Eq. Odds)',
    'acc_odds': 'Max(Acc - Eq. Opp.)'
}

for results in [mlp_results,ftl_results,hifi_results,full_results]:
    results['Performance'] = 0
    results['Fairness'] = 0
    results['Fitness Rule'] = ''
    for fitness_rule, (performance_metric, fairness_metric) in fitness_rules_target_metrics.items():
        results.loc[results.fitness_rule == fitness_rule,'Performance'] = results.loc[results.fitness_rule == fitness_rule,performance_metric]
        results.loc[results.fitness_rule == fitness_rule,'Fairness'] = results.loc[results.fitness_rule == fitness_rule,fairness_metric]
        results.loc[results.fitness_rule == fitness_rule,'Fitness Rule Abvr'] = fitness_rules_abvr[fitness_rule]
        results.loc[results.fitness_rule == fitness_rule,'Fitness Rule'] = 'Max(%s - %s)' % fitness_rules_target_metrics[fitness_rule]

/var/folders/z5/rq0dv5jj45qccc_tc39171cm0000gn/T/ipykernel_58323/2887299846.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.58517004 0.58512823 0.5840247  0.5836371  0.57625737 0.5769683
 0.58075151 0.57811729 0.57147862 0.55847176 0.56402881 0.58191431
 0.58343253 0.57740222 0.51579307 0.30958237 0.37962947 0.55899189
 0.53558872 0.2598879  0.57717035 0.50086739 0.33275221 0.37929293
 0.47729109 0.29196869 0.29037704 0.52303609 0.27480633 0.38668781
 0.50626951 0.29960979 0.4042848  0.52119218 0.2936422  0.33522388
 0.49730721 0.25112662 0.35119067 0.510052   0.26843855 0.36081008
 0.52157495 0.30114722 0.42995147 0.48267704 0.29056691 0.23139881
 0.54223416 0.23653437 0.27041017 0.52121263 0.25759348 0.23912165
 0.5223892  0.27406652 0.22303564 0.54094974 0.27699558 0.30939251
 0.51731345 0.30339828 0.37825089 0.52113734 0.26323857 0.33963196
 0.5538225  0.27850228 0.30234984 0.51877126 0.29494101 0.26489

In [6]:
datasets = ['Adult Income', 'Bank Marketing', 'Compas Recidivism','German Credit']
datasets

['Adult Income', 'Bank Marketing', 'Compas Recidivism', 'German Credit']

In [7]:
fitness_rules = ['mcc_parity', 'mcc_opportunity', 'mcc_odds', 'acc_parity', 'acc_opportunity', 'acc_odds']
fitness_rules

['mcc_parity',
 'mcc_opportunity',
 'mcc_odds',
 'acc_parity',
 'acc_opportunity',
 'acc_odds']

In [8]:
ftl_methods = ['FTL', r'FTL+SDR$_{\xi}$']
mlp_methods = ['MLP', r'MLP+SDR$_{\xi}$']
hifi_methods = ['HIFI', r'FTL+SDR$_{\xi}$']
significances = []
grouped_results_list = []

In [9]:
for path, methods, results in zip(['mlp_multi_aso_data_list.json', 'ftl_multi_aso_data_list.json', 'hifi_multi_aso_data_list.json'],
                                  [mlp_methods, ftl_methods, hifi_methods],
                                  (mlp_results, ftl_results, hifi_results)):
    method = methods[0]
    if os.path.exists(path):
        with open(path) as file:
            multi_aso_data_list = json.load(file)
    else:    
        multi_aso_data_list = []
        for d in datasets:
            for f in fitness_rules:
                
                baseline = results.loc[ (results['dataset'] == d) &
                                         (results['fitness_rule'] == f) &
                                         (results['method'] == methods[0]) ]\
                                .fitness.tolist()
                crp = results.loc[ (results['dataset'] == d) &
                                         (results['fitness_rule'] == f) &
                                         (results['method'] == methods[1]) ]\
                                .fitness.tolist()

                print(d, f, methods[0], methods[1])
                print(len(baseline), len(crp))

                # Teste bidirecional
                min_eps_forward = deepsig.aso(crp, baseline, confidence_level=0.95)
                min_eps_backward = deepsig.aso(baseline, crp, confidence_level=0.95)
                
                # Classificação com os thresholds especificados
                if min_eps_forward < 0.2:
                    interpretation = "significantly_better"
                    min_eps = min_eps_forward
                elif min_eps_forward < 0.5:
                    interpretation = "better"
                    min_eps = min_eps_forward
                elif min_eps_backward < 0.2:
                    interpretation = "significantly_worse"
                    min_eps = min_eps_backward
                elif min_eps_backward < 0.5:
                    interpretation = "worse"
                    min_eps = min_eps_backward
                else:
                    interpretation = "tie"
                    min_eps = min_eps_forward
                
                multi_aso_data_list.append({
                    'fitness_rule': f, 
                    'dataset': d, 
                    'min_eps': min_eps,
                    'min_eps_forward': min_eps_forward,
                    'min_eps_backward': min_eps_backward,
                    'interpretation': interpretation
                })
        with open(path, 'w') as file:
            json.dump(multi_aso_data_list, file)

    significance = pd.DataFrame(multi_aso_data_list)
    
    # Tabela de valores epsilon (forward e backward)
    eps_table_data = []
    for fitness_rule in fitness_rules:
        row_data = {'fitness_rule': fitness_rule}
        for dataset in datasets:
            subset = significance[(significance['fitness_rule'] == fitness_rule) & 
                                 (significance['dataset'] == dataset)]
            if not subset.empty:
                row = subset.iloc[0]
                row_data[f'{dataset}_forward'] = f"{row['min_eps_forward']:.3f}"
                row_data[f'{dataset}_backward'] = f"{row['min_eps_backward']:.3f}"
            else:
                row_data[f'{dataset}_forward'] = '-'
                row_data[f'{dataset}_backward'] = '-'
        eps_table_data.append(row_data)
    
    eps_df = pd.DataFrame(eps_table_data)
    eps_df.set_index('fitness_rule', inplace=True)
    
    # Criar MultiIndex columns para tabela de epsilon
    eps_columns = []
    for dataset in datasets:
        eps_columns.extend([
            (dataset, 'Forward'),
            (dataset, 'Backward')
        ])
    
    eps_df.columns = pd.MultiIndex.from_tuples(eps_columns)
    
    # Salvar tabela de epsilon
    eps_df.to_latex(f'tables/aso_epsilon_{method.lower()}_crp.tex')
    print(f'\n{method} - ASO Epsilon Values (Forward/Backward)')
    display(eps_df)
    
    # Também criar a tabela simplificada original
    pivot_df = significance.pivot_table(index='fitness_rule', columns='dataset', values='min_eps').sort_values(by='fitness_rule', ascending=False)
    pivot_df.to_latex(f'tables/aso_results_{method.lower()}_crp.tex')
    print(f'\n{method} - Summary ASO Results')
    display(pivot_df)

Adult Income mcc_parity MLP MLP+SDR$_{\xi}$
15 15


Bootstrap iterations: 100%|█████████▉| 999/1000 [00:00<00:00, 1227.77it/s]


Adult Income mcc_opportunity MLP MLP+SDR$_{\xi}$
15 15


Bootstrap iterations: 100%|█████████▉| 999/1000 [00:00<00:00, 1250.59it/s]


Adult Income mcc_odds MLP MLP+SDR$_{\xi}$
15 15


...

Bootstrap iterations: 100%|█████████▉| 999/1000 [00:00<00:00, 1181.53it/s]


HIFI - ASO Epsilon Values (Forward/Backward)


Adult Income          Bank Marketing           \
                     Forward Backward        Forward Backward   
fitness_rule                                                    
mcc_parity             0.000    0.997          0.000    0.998   
mcc_opportunity        0.017    1.000          0.912    1.000   
mcc_odds               0.022    1.000          0.027    1.000   
acc_parity             0.000    0.996          1.000    0.820   
acc_opportunity        0.043    1.000          1.000    0.531   
acc_odds               0.018    1.000          1.000    1.000   

                Compas Recidivism          German Credit           
                          Forward Backward       Forward Backward  
fitness_rule                                                       
mcc_parity                  0.000    0.998         0.198    1.000  
mcc_opportunity             0.022    1.000         1.000    0.511  
mcc_odds                    0.006    1.000         1.000    0.687  
acc_parity                  0.000    1.000         1.000    0.315  
acc_opportunity             0.027    1.000         0.857    1.000  
acc_odds                    0.000    0.998         1.000    0.134


HIFI - Summary ASO Results


dataset,Adult Income,Bank Marketing,Compas Recidivism,German Credit
fitness_rule,,,,
mcc_parity,0.000000,0.000000,1.223873e-07,0.197881
mcc_opportunity,0.017179,0.912036,2.242494e-02,1.000000
mcc_odds,0.022113,0.026805,5.854291e-03,1.000000
acc_parity,0.000000,1.000000,0.000000e+00,0.314754
acc_opportunity,0.042555,1.000000,2.724459e-02,0.857379
acc_odds,0.018341,1.000000,5.691066e-05,0.134259


In [10]:
# Criar tabela de interpretação comparando os 3 baselines
# Carregar todos os dados ASO
all_aso_data = {}
baseline_names = ['MLP', 'FTL', 'HIFI']

# Mapeamento de interpretação para símbolos
symbol_map = {
    'significantly_better': '++',
    'better': '+',
    'tie': '≈',
    'worse': '-',
    'significantly_worse': '--'
}

for path, baseline in zip(['mlp_multi_aso_data_list.json', 'ftl_multi_aso_data_list.json', 'hifi_multi_aso_data_list.json'],
                          baseline_names):
    with open(path) as file:
        all_aso_data[baseline] = pd.DataFrame(json.load(file))

# Criar tabela de interpretação com subcolunas por dataset
interp_table_data = []
for fitness_rule in fitness_rules:
    row_data = {'fitness_rule': fitness_rule}
    for dataset in datasets:
        for baseline in baseline_names:
            subset = all_aso_data[baseline][(all_aso_data[baseline]['fitness_rule'] == fitness_rule) & 
                                            (all_aso_data[baseline]['dataset'] == dataset)]
            if not subset.empty:
                row = subset.iloc[0]
                # Converter interpretação para símbolo
                row_data[f'{dataset}_{baseline}'] = symbol_map.get(row['interpretation'], '?')
            else:
                row_data[f'{dataset}_{baseline}'] = '-'
    interp_table_data.append(row_data)

interp_df = pd.DataFrame(interp_table_data)
interp_df.set_index('fitness_rule', inplace=True)

# Criar MultiIndex columns para tabela de interpretação
interp_columns = []
for dataset in datasets:
    for baseline in baseline_names:
        interp_columns.append((dataset, baseline))

interp_df.columns = pd.MultiIndex.from_tuples(interp_columns)

# Salvar tabela de interpretação
interp_df.to_latex('tables/aso_interpretation_comparison.tex')
print('\nASO Interpretation Comparison (All Baselines)')
display(interp_df)

# Criar tabela resumo com contagem de interpretações
summary_data = []
for baseline in baseline_names:
    counts = all_aso_data[baseline]['interpretation'].value_counts()
    summary_row = {'Baseline': baseline}
    for interp_type in ['significantly_better', 'better', 'tie', 'worse', 'significantly_worse']:
        summary_row[interp_type] = counts.get(interp_type, 0)
    summary_data.append(summary_row)

summary_df = pd.DataFrame(summary_data)
summary_df.set_index('Baseline', inplace=True)

# Renomear colunas para LaTeX
summary_df.columns = ['Sig. Better (++)', 'Better (+)', 'Tie (≈)', 'Worse (-)', 'Sig. Worse (--)']

# Adicionar linha de somatório
total_row = summary_df.sum()
total_row.name = 'Total'
summary_df = pd.concat([summary_df, total_row.to_frame().T])

# Salvar tabela resumo
summary_df.to_latex('tables/aso_interpretation_summary.tex')
print('\nASO Interpretation Summary (Global Counts)')
display(summary_df)


ASO Interpretation Comparison (All Baselines)


Adult Income          Bank Marketing           \
                         MLP FTL HIFI            MLP FTL HIFI   
fitness_rule                                                    
mcc_parity                 ≈   +   ++              ≈   -   ++   
mcc_opportunity            +   +   ++              ≈   -    ≈   
mcc_odds                  ++   +   ++              -   ≈   ++   
acc_parity                 ≈   +   ++              -   ≈    ≈   
acc_opportunity            ≈   ≈   ++             ++   ≈    ≈   
acc_odds                   +   +   ++              -   -    ≈   

                Compas Recidivism          German Credit           
                              MLP FTL HIFI           MLP FTL HIFI  
fitness_rule                                                       
mcc_parity                     ++   ≈   ++             ≈   +   ++  
mcc_opportunity                ++  ++   ++             ≈  --    ≈  
mcc_odds                       ++   ≈   ++             ≈   -    ≈  
acc_parity                      +  ++   ++             ≈   -    -  
acc_opportunity                ++   -   ++             +   ≈    ≈  
acc_odds                       ++   +   ++             ≈   -   --


ASO Interpretation Summary (Global Counts)


,Sig. Better (++),Better (+),Tie (≈),Worse (-),Sig. Worse (--)
MLP,7,4,10,3,0
FTL,2,7,7,7,1
HIFI,15,0,7,1,1
Total,24,11,24,11,2


In [11]:
grouped_results = full_results\
    .groupby(['fitness_rule', 'dataset', 'method'])\
    .agg({'fitness': ['mean', 'std', 'count'], 'Performance': ['mean', 'std'], 'Fairness': ['mean', 'std']})\
    .sort_values(by=['fitness_rule', 'dataset', ('fitness','mean')], ascending=[False, True, False])
grouped_results['formatted_fitness'] = grouped_results.apply(lambda row: f"${row[('fitness', 'mean')]:.3f} (\pm{row[('fitness', 'std')]:.2f})$", axis=1)
grouped_results['formatted_performance'] = grouped_results.apply(lambda row: f"${row[('Performance', 'mean')]:.3f} (\pm{row[('Performance', 'std')]:.2f})$", axis=1)
grouped_results['formatted_fairness'] = grouped_results.apply(lambda row: f"${row[('Fairness', 'mean')]:.3f} (\pm{row[('Fairness', 'std')]:.2f})$", axis=1)
display(grouped_results)

fitness                  \
                                                mean       std count   
fitness_rule dataset       method                                      
mcc_parity   Adult Income  FTL+SDR$_{\xi}$  0.494373  0.014363    30   
                           FTL              0.486772  0.017670    25   
                           HIFI             0.395732  0.014024    15   
                           MLP+SDR$_{\xi}$  0.387579  0.010264    15   
                           MLP              0.385655  0.010666    15   
...                                              ...       ...   ...   
acc_odds     German Credit HIFI             0.677358  0.042788    15   
                           FTL              0.668996  0.053627    13   
                           MLP+SDR$_{\xi}$  0.640351  0.062612    15   
                           FTL+SDR$_{\xi}$  0.630951  0.062479    30   
                           MLP              0.619366  0.070376    30   

                                           Performance            Fairness  \
                                                  mean       std      mean   
fitness_rule dataset       method                                            
mcc_parity   Adult Income  FTL+SDR$_{\xi}$    0.516994  0.017254  0.022621   
                           FTL                0.508716  0.021961  0.021944   
                           HIFI               0.572667  0.009679  0.176935   
                           MLP+SDR$_{\xi}$    0.578472  0.010924  0.190892   
                           MLP                0.576385  0.009136  0.190730   
...                                                ...       ...       ...   
acc_odds     German Credit HIFI               0.722333  0.033587  0.044975   
                           FTL                0.711923  0.019742  0.042927   
                           MLP+SDR$_{\xi}$    0.747667  0.020430  0.107316   
                           FTL+SDR$_{\xi}$    0.720667  0.028337  0.089716   
                           MLP                0.740000  0.031786  0.120634   

                                                      formatted_fitness  \
                                                 std                      
fitness_rule dataset       method                                         
mcc_parity   Adult Income  FTL+SDR$_{\xi}$  0.021924  $0.494 (\pm0.01)$   
                           FTL              0.019962  $0.487 (\pm0.02)$   
                           HIFI             0.014412  $0.396 (\pm0.01)$   
                           MLP+SDR$_{\xi}$  0.012732  $0.388 (\pm0.01)$   
                           MLP              0.008002  $0.386 (\pm0.01)$   
...                                              ...                ...   
acc_odds     German Credit HIFI             0.044700  $0.677 (\pm0.04)$   
                           FTL              0.065195  $0.669 (\pm0.05)$   
                           MLP+SDR$_{\xi}$  0.058156  $0.640 (\pm0.06)$   
                           FTL+SDR$_{\xi}$  0.073788  $0.631 (\pm0.06)$   
                           MLP              0.068075  $0.619 (\pm0.07)$   

                                           formatted_performance  \
                                                                   
fitness_rule dataset       method                                  
mcc_parity   Adult Income  FTL+SDR$_{\xi}$     $0.517 (\pm0.02)$   
                           FTL                 $0.509 (\pm0.02)$   
                           HIFI                $0.573 (\pm0.01)$   
                           MLP+SDR$_{\xi}$     $0.578 (\pm0.01)$   
                           MLP                 $0.576 (\pm0.01)$   
...                                                          ...   
acc_odds     German Credit HIFI                $0.722 (\pm0.03)$   
                           FTL                 $0.712 (\pm0.02)$   
                           MLP+SDR$_{\xi}$     $0.748 (\pm0.02)$   
                           FTL+SDR$_{\xi}$     $0.721 (\pm0.03)$   
        

In [12]:
selected_columns = ['formatted_fitness', 'formatted_performance', 'formatted_fairness']

for fitness_rule in fitness_rules:
    grouped_results.loc[fitness_rule][selected_columns].to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex')
     #.to_latex(f'tables/grouped_results_{fitness_rule}_crp.tex', columns=selected_columns))